# 04: Comment-Video Link
=========================

This notebook links comments with their source video metadata to provide product context for demand signal analysis.

**Key Features:**
- Merge comments with video metadata (title, description, channel, etc.)
- Classify product categories based on video titles and descriptions
- Extract video context tags (unboxing, review, tutorial, etc.)
- Calculate video engagement rates

**Output:**
- `comments_video_linked.parquet` - Full linked dataset
- `video_summary.parquet` - Aggregated video statistics

In [ ]:
import pandas as pd
from pathlib import Path
from tqdm import tqdm
import warnings
warnings.filterwarnings('ignore')

tqdm.pandas()

In [ ]:
# Paths
BASE_DIR = Path.cwd()
DATA_DIR = BASE_DIR / "../data"
OUTPUT_DIR = DATA_DIR / "linked_data"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Load data
print("Loading data...")
comments = pd.read_parquet(DATA_DIR / "shared_data/master/comments_master.parquet")
videos = pd.read_parquet(DATA_DIR / "shared_data/master/videos_master.parquet")

print(f"Comments: {len(comments):,}")
print(f"Videos: {len(videos):,}")

## 1. Merge Comments with Video Metadata

In [ ]:
print("Merging comments with video metadata...")

# Select relevant video columns
video_cols = [
    "video_id", "channel_title", "title", "description", "tags",
    "category_id", "default_language", "video_published_at",
    "view_count", "like_count", "comment_count",
]

videos_subset = videos[video_cols].drop_duplicates(subset=["video_id"])

# Merge with comments
comments_linked = comments.merge(
    videos_subset, 
    on="video_id", 
    how="left", 
    suffixes=("_comment", "_video")
)

# Rename columns for clarity
comments_linked = comments_linked.rename(columns={
    "like_count_comment": "comment_like_count",
    "like_count_video": "video_like_count",
    "comment_count": "video_comment_count"
})

print(f"Linked comments: {len(comments_linked):,}")
print(f"Missing video info: {comments_linked['title'].isna().sum()}")

## 2. Product Category Classification

In [ ]:
print("Classifying product categories...")

# Define product category keywords
CATEGORY_KEYWORDS = {
    "camera_optics": [
        "camera", "lens", "drone", "gopro", "photography", "gimbal",
        "tripod", "stabilizer", "mirrorless", "dslr", "action cam"
    ],
    "digital_accessories": [
        "cable", "charger", "power bank", "ssd", "hard drive", "usb",
        "storage", "adapter", "hub", "dock", "laptop stand", "mouse"
    ],
    "gaming": [
        "controller", "keyboard", "mouse", "headset", "gaming",
        "playstation", "xbox", "nintendo", "steam deck", "switch"
    ],
    "travel_outdoor": [
        "travel", "camping", "hiking", "backpack", "luggage",
        "outdoor", "adventure", "packing", "gear bag", "duffel"
    ],
    "collectibles": [
        "watch", "jewelry", "collectible", "figurine", "statue",
        "arcade", "model", "vinyl", "lego"
    ],
    "tools_equipment": [
        "tool", "drill", "saw", "precision", "repair", "hardware",
        "mechanical", "instrument"
    ],
    "beauty_medical": [
        "makeup", "beauty", "skincare", "medical", "health",
        "cosmetics", "brush", "dermatology"
    ],
    "audio": [
        "speaker", "microphone", "audio", "headphone", "earphone",
        "airpod", "soundbar", "amplifier", "synthesizer"
    ],
    "general_protection": [
        "case", "cover", "bag", "pouch", "sleeve", "organizer",
        "storage", "protection", "protective"
    ]
}


def classify_product_category(row):
    """Classify product category based on video title and description."""
    text = f"{row.get('title', '')} {row.get('description', '')}".lower()
    matched = []
    for category, keywords in CATEGORY_KEYWORDS.items():
        for kw in keywords:
            if kw in text:
                matched.append(category)
                break
    return "|".join(matched) if matched else "unknown"


comments_linked["product_categories"] = comments_linked.progress_apply(
    classify_product_category, axis=1
)

In [ ]:
# Category distribution
print("\nCategory distribution:")
cat_counts = comments_linked["product_categories"].str.split("|").explode().value_counts()
print(cat_counts)

## 3. Video Context Tags

In [ ]:
print("Extracting video context tags...")

def extract_video_context(row):
    """Extract content type and context from video title."""
    title = str(row.get("title", "")).lower()
    context = []

    if any(x in title for x in ["unboxing", "first look", "unbox"]):
        context.append("unboxing")
    if any(x in title for x in ["review", "honest review", "tested"]):
        context.append("review")
    if any(x in title for x in ["comparison", " vs ", "versus", "better than"]):
        context.append("comparison")
    if any(x in title for x in ["tutorial", "how to", "guide", "setup"]):
        context.append("tutorial")
    if any(x in title for x in ["haul", "what i bought", "shopping"]):
        context.append("haul")
    if any(x in title for x in ["storage", "organization", "organize", "pack"]):
        context.append("organization")

    return "|".join(context) if context else "general"


comments_linked["video_context"] = comments_linked.apply(extract_video_context, axis=1)

In [ ]:
# Video context distribution
print("\nVideo context distribution:")
ctx_counts = comments_linked["video_context"].str.split("|").explode().value_counts()
print(ctx_counts)

## 4. Video Engagement Rate

In [ ]:
print("Calculating video quality scores...")

# Engagement rate = (likes + comments) per 1000 views
comments_linked["engagement_rate"] = (
    (comments_linked["video_like_count"] + comments_linked["video_comment_count"]) /
    comments_linked["view_count"].replace(0, 1) * 1000
).round(2)

print("\nEngagement rate stats:")
print(comments_linked["engagement_rate"].describe())

## 5. Save Linked Data

In [ ]:
print("Saving linked data...")

# Save full linked dataset
comments_linked.to_parquet(OUTPUT_DIR / "comments_video_linked.parquet", index=False)
print(f"✅ Saved: comments_video_linked.parquet ({len(comments_linked):,} rows)")

# Also save as CSV for easy viewing
comments_linked.to_csv(OUTPUT_DIR / "comments_video_linked.csv", index=False)
print(f"✅ Saved: comments_video_linked.csv")

## 6. Create Video Summary Table

In [ ]:
print("Creating video summary table...")

video_summary = (
    comments_linked
    .groupby("video_id")
    .agg({
        "comment_id": "count",
        "title": "first",
        "channel_title": "first",
        "product_categories": "first",
        "video_context": "first",
        "view_count": "first",
        "video_like_count": "first",
        "video_comment_count": "first",
        "engagement_rate": "mean",
        "published_at": "min",
    })
    .reset_index()
    .rename(columns={
        "comment_id": "comment_count",
        "published_at": "first_comment_date"
    })
    .sort_values("comment_count", ascending=False)
)

video_summary.to_parquet(OUTPUT_DIR / "video_summary.parquet", index=False)
print(f"✅ Saved: video_summary.parquet ({len(video_summary):,} videos)")

In [ ]:
# Show top commented videos
print("\nTop 10 most commented videos:")
cols = ["title", "channel_title", "comment_count", "view_count"]
display(video_summary[cols].head(10))

## Summary

In [ ]:
print("="*60)
print("SUMMARY")
print("="*60)
print(f"Total linked comments: {len(comments_linked):,}")
print(f"Unique videos: {comments_linked['video_id'].nunique():,}")
print(f"\nProduct categories distribution:")
print(cat_counts)
print(f"\nVideo context distribution:")
print(ctx_counts)
print(f"\nData saved to: {OUTPUT_DIR}")
print("="*60)